In [6]:
from models.model import ModelTrainer
import requests
import json
from collections import Counter
import pandas as pd

base_url = "https://api-base-fastapi-mpt-dkhjb0cydga2aydc.canadacentral-01.azurewebsites.net"
api_key = ""
with open('../api.json', 'r') as file:
    api_key = json.load(file)["GH_API_KEY"]

def get_call(paths,url=base_url, params=None, header=None):
    url = f"{base_url}/{paths}"
    response = requests.get(url, params=params, headers=header)
    if response.status_code == 200:
        return response.json()
    elif response.status_code == 403:
        print("La patience est d'or. Enculé")
        response = requests.get(url, params=params, headers=header)
        return response.json()
    else:
        print(f"Error: {response.status_code} - {response.json().get('message', 'Unknown error')}")
        return None

repositories = get_call(paths="show_data",params={"page_size":100,"page":4})

ProxyError: HTTPSConnectionPool(host='api-base-fastapi-mpt-dkhjb0cydga2aydc.canadacentral-01.azurewebsites.net', port=443): Max retries exceeded with url: /show_data?page_size=100&page=4 (Caused by ProxyError('Unable to connect to proxy', NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x000001D7E88C49D0>: Failed to resolve 'cosmos2.mc2.renault.fr' ([Errno 11001] getaddrinfo failed)")))

In [11]:
df = pd.DataFrame(repositories)
df

,page,page_size,total_pages,total_documents,data
0,1,10,14619,146188,"{'name': 'Test', 'value': '42'}"
1,1,10,14619,146188,"{'id': 2922981, 'node_id': 'MDEwOlJlcG9zaXRvcn..."
2,1,10,14619,146188,"{'id': 18222014, 'node_id': 'MDEwOlJlcG9zaXRvc..."
3,1,10,14619,146188,"{'id': 22457583, 'node_id': 'MDEwOlJlcG9zaXRvc..."
4,1,10,14619,146188,"{'id': 41860912, 'node_id': 'MDEwOlJlcG9zaXRvc..."
5,1,10,14619,146188,"{'id': 21751038, 'node_id': 'MDEwOlJlcG9zaXRvc..."
6,1,10,14619,146188,"{'id': 48772800, 'node_id': 'MDEwOlJlcG9zaXRvc..."
7,1,10,14619,146188,"{'id': 17987852, 'node_id': 'MDEwOlJlcG9zaXRvc..."
8,1,10,14619,146188,"{'id': 23153517, 'node_id': 'MDEwOlJlcG9zaXRvc..."
9,1,10,14619,146188,"{'id': 29298170, 'node_id': 'MDEwOlJlcG9zaXRvc..."


In [ ]:
import mlflow
import mlflow.sklearn
from models.model_factory import get_model
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import pandas as pd

# Charger les données
repositories = [...]  # Charger tes données ici
model_name = "gradient_boosting"  # 🔥 Change ici pour tester d'autres modèles

# Charger le modèle depuis la factory
model = get_model(model_name, lr=0.01, n_estimators=100)
df = model.prepare_data(repositories)

X = df.drop(columns=["language"])
y = df["language"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, shuffle=True)

# Expérimentation MLflow
with mlflow.start_run():
    mlflow.log_param("model_name", model_name)
    
    model.train(X_train, y_train)
    y_pred = model.predict(X_test)

    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    mlflow.log_metric("mse", mse)
    mlflow.log_metric("r2_score", r2)

    mlflow.sklearn.log_model(model.model, f"{model_name}_model")

    print(f"📉 MSE: {mse:.4f} | 📈 R² Score: {r2:.4f}")

# Sauvegarder le modèle localement
model.save_model(f"models/{model_name}.pkl")
